# Modelo base para comparar objetivo

In [130]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import pandas as pd


from sklearn.dummy import DummyRegressor
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient


print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:../mlruns


In [131]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("HousePrices")

with mlflow.start_run(run_name="init"):
    mlflow.log_metric("test", 1)

In [132]:
X_train = np.load("../data/processed/X_train.npy")
y_train = np.load("../data/processed/y_train.npy")

X_test = np.load("../data/processed/X_test.npy")
y_test = np.load("../data/processed/y_test.npy")

In [133]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
print(y_train[:5])

(1168, 229) (1168,)
(292, 229) (292,)
[11.88449592 12.08954445 11.3504183  12.07254697 11.75195024]


In [134]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def rmse_dollars(y_true_log, y_pred_log):
    return float(np.sqrt(mean_squared_error(np.exp(y_true_log), np.exp(y_pred_log))))

def mape_dollars(y_true_log, y_pred_log):
    return float(mean_absolute_percentage_error(
        np.exp(y_true_log),
        np.exp(y_pred_log)
    ))

def evaluate_predictions(y_true, y_pred):
    return {
        "rmse_log": rmse(y_true, y_pred),
        "rmse_dollars": rmse_dollars(y_true, y_pred),
        "mape": mape_dollars(y_true, y_pred),
    }

In [135]:
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)

baseline_metrics = evaluate_predictions(y_test, y_pred)

In [136]:
def log_to_mlflow(run_name, model, metrics, params):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model")

In [137]:
log_to_mlflow(
    run_name="baseline_mean_processed",
    model=baseline_model,
    metrics=baseline_metrics,
    params={
        "model": "DummyRegressor",
        "strategy": "mean",
        "target": "log(SalePrice)",
        "dataset": "processed"
    }
)

2026/04/13 03:44:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 03:44:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [138]:
baseline_results_df = pd.DataFrame([{
    "Modelo": "Baseline (media log)",
    "RMSE (log)": baseline_metrics["rmse_log"],
    "RMSE ($)": baseline_metrics["rmse_dollars"],
    "MAPE (%)": baseline_metrics["mape"] * 100,
}])

display(
    baseline_results_df.style
    .format({
        "RMSE (log)": "{:.4f}",
        "RMSE ($)": "{:,.0f}",
        "MAPE (%)": "{:.2f}%"
    })
    .hide(axis="index")
)

Modelo,RMSE (log),RMSE ($),MAPE (%)
Baseline (media log),0.4332,"88,271",36.81%
